In [ ]:
# %pip install ninja ipykernel ipywidgets huggingface_hub --break-system-packages

In [ ]:
# %pip install --upgrade transformers --break-system-packages

In [ ]:
# %pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [ ]:
# %pip install compressed-tensors --break-system-packages

In [1]:
import os

import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from transformers import AutoModelForCausalLM, AutoProcessor


In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.13.0+cu130
CUDA Available: True
CUDA Version: 13.0
GPU Name: NVIDIA H200
VRAM: 139.8 GB


In [4]:
notebook_login()

In [5]:
MODEL_ID = "dots-studio/dots.mocr"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [6]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 22 files: 100%|██████████████████████| 22/22 [00:00<00:00, 479.54it/s]
Download complete: :                                       |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            ✓ Downloaded
  path: /workspace/local_model
Download complete: :                                       |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            


In [7]:
model = AutoModelForCausalLM.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16, 
    device_map="auto",
    trust_remote_code=True
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/643 [00:00<?, ?it/s]

In [8]:
model

DotsOCRForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rot

In [9]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:  # noqa: BLE001
        print(f"[Hub] ❌ Error uploading: {e}")

In [ ]:
TUNING_CONFIG = {
    "group_size": 16,
    "sym": True,
    "iters": 800,                  # High accuracy (Production grade)
    "nsamples": 512,               # More calibration data
    "batch_size": 4,  
    "seqlen": 2048*4,
    "low_gpu_mem_usage": False,     # Keep on GPU for speed
    "enable_torch_compile": True,   # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
}

In [14]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="NVFP4",
    **TUNING_CONFIG,
)

2026-09-10 02:41:29 WARNING autoround.py L595: Passing 'group_size' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-09-10 02:41:29 WARNING autoround.py L595: Passing 'sym' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-09-10 02:41:29 WARNING autoround.py L595: Passing 'iters' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.


In [15]:
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="llm_compressor", inplace=True
)

2026-09-10 02:41:30 INFO base.py L1352: `torch.compile` is disabled, as activation is static


2026-09-10 02:41:30 INFO composer.py L205: Block-forward torch.compile is disabled because at least one quantized layer uses NVFP4.
2026-09-10 02:41:30 INFO orchestrator.py L587: start to cache block inputs
2026-09-10 02:41:30 INFO mllm.py L86: Using MLLM template: dots_ocr
2026-09-10 02:41:30 INFO mllm.py L125: Multimodal model with non-MLLM calibration dataset 'NeelNanda/pile-10k' and quant_nontext_module=False: using the standard text dataloader (vision/audio towers are not being quantized, so text-only calibration through the full-model forward is sufficient).
2026-09-10 02:41:30 INFO calib_dataset.py L1116: Preprocessing calibration dataset in a subprocess to avoid memory leaks...
2026-09-10 02:41:37 WARNING mllm.py L212: Insufficient number of samples: required 512, but only 336 were processed.
2026-09-10 02:41:40 INFO device.py L1560: 'peak_ram': 18.52GB, 'peak_vram': 6.19GB
2026-09-10 02:41:40 INFO orchestrator.py L619: caching done
Quantizing model.layers.0:   0%|          | 0

(DotsOCRForCausalLM(
   (model): Qwen2Model(
     (embed_tokens): Embedding(151936, 1536)
     (layers): ModuleList(
       (0-27): 28 x Qwen2DecoderLayer(
         (self_attn): Qwen2Attention(
           (q_proj): QuantLinear()
           (k_proj): QuantLinear()
           (v_proj): QuantLinear()
           (o_proj): QuantLinear()
         )
         (mlp): Qwen2MLP(
           (gate_proj): QuantLinear()
           (up_proj): QuantLinear()
           (down_proj): QuantLinear()
           (act_fn): SiLUActivation()
         )
         (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
         (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
       )
     )
     (norm): Qwen2RMSNorm((1536,), eps=1e-06)
     (rotary_emb): Qwen2RotaryEmbedding()
   )
   (lm_head): Linear(in_features=1536, out_features=151936, bias=False)
   (vision_tower): DotsVisionTransformer(
     (patch_embed): DotsViTPreprocessor(
       (patchifier): DotsPatchEmbed(
         (proj): Conv2d(3, 1536, ker

In [16]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [18]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-nvfp-w4g16"), 
        f"{base_name}-AutoRound-NVFP4", 
        hf_token)
else:
    print("No Hugging Face token found. Skipping upload to hub.")


[Hub] Pushing ./AutoRound/local_model-nvfp-w4g16 to Vishva007/dots.mocr-AutoRound-NVFP4...


[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/dots.mocr-AutoRound-NVFP4
